# 🔬 Microsoft Phi-3 Mini (3.8B Instruct) — Benchmark Evaluation

**Benchmarks:**
- 📚 **CommonsenseQA** — Commonsense Reasoning (5-way MCQ)
- 🧠 **StrategyQA** — Multi-Hop Logical Reasoning (Yes/No)

**Compatible with:** Kaggle Free Tier (T4 GPU) & Google Colab Free Tier (T4 GPU)

> **Model:** `microsoft/Phi-3-mini-4k-instruct` (3.8B params) loaded in **4-bit quantization** to fit within ~15GB VRAM.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 📦 Step 1 — Install Dependencies

In [6]:
from huggingface_hub import login
login("")
print("hugging fce validation complete")

hugging fce validation complete


## 🔍 Step 2 — Check GPU Availability

In [7]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU detected: {gpu_name}")
    print(f"   Total VRAM: {total_mem:.1f} GB")
else:
    print("⚠️  No GPU detected — running on CPU (will be very slow).")
    print("   Go to Runtime > Change runtime type > GPU (T4) in Colab.")
    print("   Or enable GPU in Kaggle: Settings > Accelerator > GPU T4.")

✅ GPU detected: Tesla T4
   Total VRAM: 15.6 GB


## 🤖 Step 3 — Load Phi-3 Mini 3.8B Instruct (4-bit Quantized)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "microsoft/phi-3-mini-4k-instruct"

print(f"📥 Loading tokenizer from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"📥 Loading model in 4-bit (this may take 2-4 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype=torch.bfloat16,
)
model.eval()
print("✅ Model loaded successfully!")

📥 Loading tokenizer from: microsoft/phi-3-mini-4k-instruct


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model loaded successfully!


## 🛠️ Step 4 — Define Inference Helper

In [9]:
# !pip install -q trl

In [10]:

# #%%
# from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
# from peft import LoraConfig, get_peft_model, TaskType
# from trl import SFTTrainer, SFTConfig
# import torch
# from peft import PeftModel
# import gc

# # Load LoRA adapter and merge
# peft_model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/tunned_weight')
# merged_model = peft_model.merge_and_unload()  # Merges LoRA into base weights
# merged_model.eval()

# # This is now our inference model — replace the variable name for clarity
# model = merged_model

# print(f'✅ Fine-tuned model ready for inference.')
# print(f'   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [11]:
import torch
from collections import Counter

# --------------------------------------------------
# Base generation function (greedy by default)
# --------------------------------------------------
def generate_response(
    prompt: str,
    max_new_tokens: int = 64,
    temperature: float | None = None,
    do_sample: bool = False,
    top_p: float | None = None,
) -> str:
    """
    Formats prompt using Phi-3 chat template and generates a response.
    Greedy decoding by default (for reproducibility).
    """

    messages = [{"role": "user", "content": prompt}]

    # Apply Phi-3 instruct chat template
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response

print("✅ Self-consistency inference pipeline ready!")

✅ Self-consistency inference pipeline ready!


---
# 📚 Benchmark 1: CommonsenseQA — Commonsense Reasoning

CommonsenseQA is a 5-choice multiple choice QA dataset that requires commonsense knowledge to answer questions correctly.

- **Format:** Question + 5 answer choices (A–E) → model selects the correct letter
- **Dataset:** `tau/commonsense_qa` from HuggingFace
- **Metric:** Accuracy (%)

In [12]:
from datasets import load_dataset

print("📥 Loading CommonsenseQA validation split...")
csqa_dataset = load_dataset("tau/commonsense_qa", split="validation")
print(f"✅ Loaded {len(csqa_dataset)} validation examples.")

📥 Loading CommonsenseQA validation split...
✅ Loaded 1221 validation examples.


In [13]:
def build_csqa_prompt(question: str, choices: dict) -> str:
    """
    Builds a clear MCQ prompt for CommonsenseQA.
    choices is a dict with 'label' (list) and 'text' (list).
    """
    options_text = ""
    for label, text in zip(choices["label"], choices["text"]):
        options_text += f"\n({label}) {text}"

    # prompt = (
    #     f"Answer the following commonsense reasoning question by selecting the best answer.\n"
    #     f"Respond with ONLY the letter of your answer (A, B, C, D, or E). Do not explain.\n\n"
    #     f"Question: {question}\n"
    #     f"Options:{options_text}\n\n"
    #     f"Answer:"
    # )
    prompt = (
        "Answer the following commonsense reasoning question.\n"
        "First, reason through the options step by step.\n"
        "Then on the last line write: 'Answer: X' where X is the letter.\n\n"
        f"Question: {question}\nOptions: ..."
    )
    return prompt


def extract_csqa_answer(response: str) -> str:
    """
    Extracts the predicted letter (A-E) from the model's raw response.
    """
    response = response.strip().upper()
    # Look for first occurrence of A-E
    for char in response:
        if char in "ABCDE":
            return char
    return "X"  # No valid answer found


# Test the prompt builder
print("Complete")


Complete


## 5 Shot + CoT

In [ ]:
FEW_SHOT_EXAMPLES = """Question: What do people use to absorb extra ink from a fountain pen?
Options:
(A) shirt pocket
(B) blotting paper
(C) cup
(D) television
(E) desk
Answer: B (a shirt pocket holds pens but doesn't absorb ink, a cup holds liquid but isn't absorbent, blotting paper is a soft absorbent paper made specifically to soak up wet ink from a page, so blotting paper is the correct tool)

Question: What home appliance is never on wheels?
Options:
(A) ice cream truck
(B) washing machine
(C) ceiling fan
(D) dishwasher
(E) solar panel
Answer: C (washing machines and dishwashers sit on the floor and could theoretically have wheels, but a ceiling fan is permanently fixed to the ceiling by design and physically cannot have wheels, making it the only option that is never on wheels)

Question: The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?
Options:
(A) ignore
(B) enforce
(C) authoritarian
(D) yell at
(E) avoid
Answer: A (the school made efforts to change, but sanctions were still applied as punishment, meaning the authority imposing sanctions did not acknowledge or consider those efforts, which means they ignored them)

Question: Sammy wanted to go to where the people were. Where might he go?
Options:
(A) race track
(B) populated areas
(C) the desert
(D) apartment
(E) roadblock
Answer: B (a race track has people only during events, a desert is largely empty, an apartment has only a few people, a roadblock is temporary, but populated areas are specifically defined as places where large numbers of people consistently live and gather, making it the best answer)

Question: To locate a bug, the programmer would need to do what to the code?
Options:
(A) delete
(B) debug
(C) silence
(D) write
(E) ask
Answer: B (deleting code removes it entirely, writing adds new code, silencing suppresses errors without fixing them, but debugging is the systematic process of reading through and testing code specifically to find and fix errors, making it the correct answer)

"""

def build_csqa_prompt(question: str, choices: dict) -> str:
    options_text = ""
    for label, text in zip(choices["label"], choices["text"]):
        options_text += f"\n({label}) {text}"

    prompt = (
        f"Answer commonsense reasoning questions. Study the examples below carefully, then answer the final question.\n"
        f"All choices (A, B, C, D, and E) are equally likely to be correct.\n"
        f"First think through why the other options are wrong, then give the correct letter followed by your reasoning in parentheses.\n\n"
        f"{FEW_SHOT_EXAMPLES}"
        f"Question: {question}\n"
        f"Options:{options_text}\n\n"
        f"Answer:"
    )
    return prompt


def extract_csqa_answer(response: str) -> str:
    response = response.strip().upper()
    for char in response:
        if char in "ABCDE":
            return char
    return "X"


print("Complete")

In [14]:
from collections import Counter

def generate_response_self_consistent_csqa(
    prompt,
    num_samples=5,
    max_new_tokens=10,
    temperature=0.7,
):
    """
    Runs multiple sampled generations and returns:
    - majority-voted answer (A/B/C/D/E)
    - all raw outputs (for debugging)
    """
    votes = []
    raw_outputs = []

    for _ in range(num_samples):
        out = generate_response(
            prompt=prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,   # 🔑 required for self-consistency
            top_p=0.9,
        )

        raw_outputs.append(out)

        pred = extract_csqa_answer(out)
        if pred is not None:
            votes.append(pred)

    if len(votes) == 0:
        return None, raw_outputs

    majority_pred = Counter(votes).most_common(1)[0][0]
    return majority_pred, raw_outputs

## Raw Model

In [15]:
import time
from tqdm.auto import tqdm

# ─── CONFIG ───────────────────────────────────────────────
# Set NUM_SAMPLES to None to run on the full validation set (~1221 examples)
# For a quick test on free tier, we use 200 samples (~10-15 min on T4)
NUM_SAMPLES = 500
# ──────────────────────────────────────────────────────────

csqa_eval = csqa_dataset.select(range(NUM_SAMPLES)) if NUM_SAMPLES else csqa_dataset

csqa_results = []
csqa_correct = 0

print(f"🚀 Evaluating on {len(csqa_eval)} CommonsenseQA examples...")
start_time = time.time()

for i, example in enumerate(tqdm(csqa_eval, desc="CommonsenseQA")):
    prompt   = build_csqa_prompt(example["question"], example["choices"])
    raw_out  = generate_response(prompt, max_new_tokens=40)
    pred     = extract_csqa_answer(raw_out)
    gold     = example["answerKey"].strip().upper()
    correct  = (pred == gold)

    if correct:
        csqa_correct += 1

    csqa_results.append({
        "question": example["question"],
        "gold": gold,
        "predicted": pred,
        "raw_output": raw_out,
        "correct": correct,
    })

elapsed = time.time() - start_time
csqa_accuracy = csqa_correct / len(csqa_eval) * 100

print(f"\n{'='*50}")
print(f"📊 CommonsenseQA Results")
print(f"{'='*50}")
print(f"  Samples evaluated : {len(csqa_eval)}")
print(f"  Correct           : {csqa_correct}")
print(f"  Accuracy          : {csqa_accuracy:.2f}%")
print(f"  Time taken        : {elapsed/60:.1f} minutes")
print(f"{'='*50}")

🚀 Evaluating on 500 CommonsenseQA examples...


CommonsenseQA:   0%|          | 0/500 [00:00<?, ?it/s]


📊 CommonsenseQA Results
  Samples evaluated : 500
  Correct           : 368
  Accuracy          : 73.60%
  Time taken        : 35.1 minutes


## Raw model with self consistency

In [ ]:
import time
from tqdm.auto import tqdm

# ─── CONFIG ───────────────────────────────────────────────
NUM_SAMPLES = 500
SC_K = 5  # number of self-consistency samples
# ──────────────────────────────────────────────────────────

csqa_eval = csqa_dataset.select(range(NUM_SAMPLES)) if NUM_SAMPLES else csqa_dataset

csqa_results = []
csqa_correct = 0

print(f"🚀 Evaluating on {len(csqa_eval)} CommonsenseQA examples with Self-Consistency (k={SC_K})...")
start_time = time.time()

for i, example in enumerate(tqdm(csqa_eval, desc="CommonsenseQA")):
    prompt = build_csqa_prompt(example["question"], example["choices"])

    pred, raw_outs = generate_response_self_consistent_csqa(
        prompt,
        num_samples=SC_K,
        max_new_tokens=10,
        temperature=0.7,
    )

    gold = example["answerKey"].strip().upper()
    correct = (pred == gold)

    if correct:
        csqa_correct += 1

    csqa_results.append({
        "question": example["question"],
        "gold": gold,
        "predicted": pred,
        "raw_outputs": raw_outs,   # ← multiple generations
        "correct": correct,
    })

elapsed = time.time() - start_time
csqa_accuracy = csqa_correct / len(csqa_eval) * 100

print(f"\n{'='*50}")
print(f"📊 CommonsenseQA Results (Self-Consistency)")
print(f"{'='*50}")
print(f"  Samples evaluated : {len(csqa_eval)}")
print(f"  Correct           : {csqa_correct}")
print(f"  Accuracy          : {csqa_accuracy:.2f}%")
print(f"  Time taken        : {elapsed/60:.1f} minutes")
print(f"{'='*50}")

🚀 Evaluating on 500 CommonsenseQA examples with Self-Consistency (k=5)...


CommonsenseQA:   0%|          | 0/500 [00:00<?, ?it/s]


📊 CommonsenseQA Results (Self-Consistency)
  Samples evaluated : 500
  Correct           : 105
  Accuracy          : 21.00%
  Time taken        : 31.0 minutes


In [ ]:
    # Per-label breakdown
import pandas as pd
print("\n📊 Accuracy by gold answer label:")
df_csqa = pd.DataFrame(csqa_results)
label_stats = df_csqa.groupby("gold")["correct"].agg(["sum", "count"])
label_stats["accuracy"] = (label_stats["sum"] / label_stats["count"] * 100).round(2)
label_stats.columns = ["Correct", "Total", "Accuracy (%)"]
print(label_stats)


📊 Accuracy by gold answer label:
      Correct  Total  Accuracy (%)
gold                              
A          77     94         81.91
B          79    106         74.53
C          78    102         76.47
D          73     94         77.66
E          73    104         70.19


In [ ]:
# Per-label breakdown
import pandas as pd
print("\n📊 Accuracy Tunned Model:")
df_csqa = pd.DataFrame(csqa_results)
label_stats = df_csqa.groupby("gold")["correct"].agg(["sum", "count"])
label_stats["accuracy"] = (label_stats["sum"] / label_stats["count"] * 100).round(2)
label_stats.columns = ["Correct", "Total", "Accuracy (%)"]
print(label_stats)


📊 Accuracy Tunned Model:
      Correct  Total  Accuracy (%)
gold                              
A          72     94         76.60
B          80    106         75.47
C          75    102         73.53
D          75     94         79.79
E          74    104         71.15


In [ ]:
# Per-label breakdown
import pandas as pd
print("\n📊 Accuracy Tunned Model:")
df_csqa = pd.DataFrame(csqa_results)
label_stats = df_csqa.groupby("gold")["correct"].agg(["sum", "count"])
label_stats["accuracy"] = (label_stats["sum"] / label_stats["count"] * 100).round(2)
label_stats.columns = ["Correct", "Total", "Accuracy (%)"]
print(label_stats)


📊 Accuracy Tunned Model:
      Correct  Total  Accuracy (%)
gold                              
A          77     94         81.91
B          79    106         74.53
C          78    102         76.47
D          73     94         77.66
E          73    104         70.19


In [17]:
# Per-label breakdown
import pandas as pd
print("\n📊 Accuracy Tunned Model:")
df_csqa = pd.DataFrame(csqa_results)
label_stats = df_csqa.groupby("gold")["correct"].agg(["sum", "count"])
label_stats["accuracy"] = (label_stats["sum"] / label_stats["count"] * 100).round(2)
label_stats.columns = ["Correct", "Total", "Accuracy (%)"]
print(label_stats)


📊 Accuracy Tunned Model:
      Correct  Total  Accuracy (%)
gold                              
A          72     94         76.60
B          64    106         60.38
C          72    102         70.59
D          73     94         77.66
E          87    104         83.65
